In [1]:
import os
import sys
import io
import json
import time
import regex as re
import xmltodict
import xml.etree.ElementTree as ET
from pymed import PubMed


In [8]:
# List of journals to include
journals_to_include = [
    "ACS Applied Materials & Interfaces",
    "ACS Nano",
    "Advanced Functional Materials",
    "Advanced Materials",
    "Angewandte Chemie",
    "Biology and Medicine",
    "Biomaterials",
    "Cell",
    "Clinical Cancer Research",
    "Frontiers in Nanotechnology",
    "Immunity",
    "International Journal of Nanomedicine",
    "Journal of Controlled Release",
    "Journal of Materials Chemistry B",
    "Matter",
    "Molecular Therapy",
    "Nano Letters",
    "Nano Micro Small",
    "Nano Research",
    "Nanomedicine",
    "Nanomedicine: Nanotechnology",
    "Nanoscale",
    "Nature",
    "Nature Biomedical Engineering",
    "Nature Cancer",
    "Nature Communications",
    "Nature Materials",
    "Nature Medicine",
    "Nature Nanotechnology",
    "NPG Asia Materials",
    "Pharmaceutics",
    "PNAS",
    "Science",
    "Science Advances",
    "Science Translational Medicine",
    "Scientific Reports",
    "Small"
]

# Inclusion criteria
keywords_inclusion = ["nanoparticles", "nanoparticle", "in vivo", "mouse model", "animal model"]

# Exclusion criteria
keywords_exclusion = ["review"]


In [17]:
# Create a PubMed object that GraphQL can use to query
# Note that the parameters are not required but kindly requested by PubMed Central
# https://www.ncbi.nlm.nih.gov/pmc/tools/developers/
pubmed = PubMed(tool="MyRetrivalTool", email="c.g.a.viviers@tue.nl")

# Create a GraphQL query in plain text

# Example
# query = '(("2018/05/01"[Date - Create] : "3000"[Date - Create])) AND (Xiaoying Xian[Author] OR diabetes)'

query = '("nanoparticles"[Title] OR "nanoparticle"[Title]) AND ("in vivo"[Title] OR "mouse model"[Title] OR "animal model"[Title])'

# Execute the query against the API
results = pubmed.query(query, max_results=50)

In [18]:
results

In [19]:
def format_filename(input_string, max_length=100):
    """
    Format a string to be used as a filename.
    
    Args:
    input_string (str): The string to be formatted.
    max_length (int): Maximum length of the resulting filename (default: 100).
    
    Returns:
    str: A formatted string suitable for use as a filename.
    """
    # Convert to lowercase
    filename = input_string.lower()
    
    # Replace spaces with underscores
    filename = filename.replace(' ', '_')
    
    # Remove any non-alphanumeric characters (except underscores and hyphens)
    filename = re.sub(r'[^\w-]', '', filename)
    
    # Replace consecutive underscores or hyphens with a single underscore
    filename = re.sub(r'[_-]+', '_', filename)
    
    # Remove leading and trailing underscores
    filename = filename.strip('_')
    
    # Truncate to the specified maximum length
    filename = filename[:max_length]
    
    # Ensure the filename is not empty after processing
    if not filename:
        filename = "untitled"
    
    return filename

In [20]:
def clean_for_json(obj):
    if isinstance(obj, str):
        return obj.encode('ascii', 'ignore').decode('ascii')
    elif isinstance(obj, dict):
        return {clean_for_json(key): clean_for_json(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [clean_for_json(element) for element in obj]
    else:
        return obj

In [21]:
failed_articles = []

# Write each article to a file

# Create a directory to store the articles
if not os.path.exists('articles'):
    os.makedirs('articles')

# Loop over the retrieved articles
for article in results:

    # Extract and format all the information from the article
    article_dict = article.toDict()

    # check if article is in journals_to_include, make all case insensitive
    if article_dict['journal'].lower() not in [journal.lower() for journal in journals_to_include]:
        print(f'Article {article_dict["title"]} is not in the journals to include \n Journal: {article_dict["journal"]}')
        continue

    # make sure all the data is json serializable, 
    for key in article_dict.keys():
        if not isinstance(article_dict[key], (str, int, float, list, dict)):
           # if xml object, convert to string
            if isinstance(article_dict[key], ET.Element):
                article_dict[key] = ET.tostring(article_dict[key], encoding='utf8').decode('utf8')
            # if xml string, convert to dict
            elif isinstance(article_dict[key], str) and article_dict[key].startswith('<'):
                article_dict[key] = xmltodict.parse(article_dict[key])
            else:
           
             article_dict[key] = str(article_dict[key])

    
             
    title = format_filename(article_dict['title'])
    
    file_name = f'articles/{title}.json'

    try:
        # cleaned_article_dict = clean_for_json(article_dict)

        with open(file_name, 'w', encoding='utf-8') as file:
            json.dump(article_dict, file, ensure_ascii=True, indent=4)

    except Exception as e:
        print(f'Error writing file {file_name}')
        print(e)
        failed_articles.append(file_name)


Article Antibiotic-loaded nanoparticles for the treatment of intracellular methicillin-resistant Staphylococcus Aureus infections: In vitro and in vivo efficacy of a novel antibiotic. is not in the journals to include 
 Journal: Journal of controlled release : official journal of the Controlled Release Society
Article  is not in the journals to include 
 Journal: ACS sensors
Article Enhancing orthodontic treatment control with fish scale-derived hydroxyapatite nanoparticles: Insights from an animal model study. is not in the journals to include 
 Journal: The Saudi dental journal
Article Coating based on chitosan/vancomycin nanoparticles: Patterns of formation in a water-carbon dioxide biphase system and in vivo stability. is not in the journals to include 
 Journal: International journal of biological macromolecules
Article Supramolecular assembly of polycation/mRNA nanoparticles and in vivo monocyte programming. is not in the journals to include 
 Journal: Proceedings of the National

In [ ]:
# loop through failed articles check if all have abstracts
local_results =  os.listdir('articles')


for article in local_results:
    article_path = f'articles/{article}'
    with open(article_path, 'r') as file:
        data = json.load(file)
        if 'abstract' not in data.keys():
            print(f"Article {article} does not have an abstract")
            # failed_articles.append(article)

    